In [13]:
# Multi image classification
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # allow multiple libiomp5md
os.environ["OMP_NUM_THREADS"] = "1"           # keep OpenMP under control

import torch
import albumentations as A
import numpy as np
import pandas as pd
#from .src.model import FoundationalCVModel, FoundationalCVModelWithClassifier
from fundus_dataset import FundusDataset
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
from model import UnifiedBackbone
from fundus_dataset import FundusDataset
#from .src.model import FoundationalCVModel, FoundationalCVModelWithClassifier
from torch.utils.data import DataLoader

In [14]:
model_name = "retfound_green"
run_id = 4
dataset = "mBRSET"

In [17]:
data_dir = r'C:\\Users\\preet\\Documents\\mBRSET\\mBRSET_image_quality\\'

if dataset == "BRSET":
    train_df = pd.read_pickle(data_dir + "data\\brset_train_2826.pkl")
    train_df = train_df.rename(columns={"patient_id": "patient"})
    val_df = pd.read_pickle(data_dir + "data\\brset_val_2826.pkl")
    val_df = val_df.rename(columns={"patient_id": "patient"})
    test_df = pd.read_pickle(data_dir + "data\\brset_test_2826.pkl")
    test_df = test_df.rename(columns={"patient_id": "patient"})
    str_prefix="IQ_BRSET"
    
if dataset == "mBRSET":
    train_df = pd.read_pickle(os.path.join(data_dir + 'data\\mbrset_icdr_quality_2826_train_full.pkl'))
    val_df= pd.read_pickle(os.path.join(data_dir + 'data\\mbrset_icdr_quality_2826_val_full.pkl'))
    test_df= pd.read_pickle(os.path.join(data_dir + 'data\\mbrset_icdr_quality_2826_test_full.pkl'))
    str_prefix = "IQ_mBRSET"
# Remove any rows for which final_icdr is NaN
train_df.dropna(subset=['final_icdr'], inplace=True)
val_df.dropna(subset=['final_icdr'], inplace=True)
test_df.dropna(subset=['final_icdr'], inplace=True)


In [18]:
train_df.head()

,Unnamed: 0,patient,file,final_icdr,final_quality,laterality
0,0,1,1.1.jpg,True,1,True
1,1,1,1.2.jpg,True,1,True
2,2,1,1.3.jpg,True,1,False
3,3,1,1.4.jpg,True,1,False
4,4,4,4.1.jpg,True,1,True


In [19]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

if model_name != "retfound_green":
    train_tf = A.Compose([
        A.RandomResizedCrop(height=392, width=392, scale=(0.7, 1.0), ratio=(0.75, 1.33)),
        A.HorizontalFlip(),
        A.VerticalFlip(),
        A.Normalize(mean=(0.485, 0.456, 0.406), 
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    
    val_tf = A.Compose([
        A.Resize(392, 392),
        A.Normalize(mean=(0.485, 0.456, 0.406), 
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
else:
    train_tf = A.Compose([
        A.RandomResizedCrop(height=392, width=392, scale=(0.7, 1.0), ratio=(0.75, 1.33)),
        A.HorizontalFlip(),
        A.VerticalFlip(),
        A.Normalize(mean=(0.5, 0.5, 0.5), 
                    std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ])
    
    val_tf = A.Compose([
        A.Resize(392, 392),
        A.Normalize(mean=(0.5, 0.5, 0.5), 
                    std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ])

In [20]:
if dataset == "BRSET":
    img_root = r"C:\Users\preet\Documents\BRSET\data\resized_fundus_photos"

if dataset =="mBRSET":
    img_root = r"C:\\Users\\preet\\Documents\\mBRSET\\mbrset-a-mobile-brazilian-retinal-dataset-1.0\\images"

train_dataset = FundusDataset(
    df=train_df,
    img_root=img_root,
    high_quality_tf=train_tf,
    low_quality_tf=train_tf,
    label_col='final_quality'
)

val_dataset  = FundusDataset(val_df,   img_root=img_root, high_quality_tf=val_tf, low_quality_tf = val_tf,label_col='final_quality'
)
test_dataset  = FundusDataset(test_df,  img_root=img_root, high_quality_tf=val_tf, low_quality_tf = val_tf,label_col='final_quality')
print(train_dataset)

In [21]:
train_df["final_quality"].value_counts()

final_quality
1    2799
0     105
Name: count, dtype: int64

In [22]:
from img_quality_train_val import validate, train_model


In [23]:
import numpy as np
import torch
import torch.nn as nn
from img_quality_train_val import validate, find_thresholds_for_recall, show_conf_matrix_at_threshold
def compute_class_weights(df, threshold=0):
    labels = (df["final_quality"] > threshold).astype(int).values
    class_counts = np.bincount(labels)
    weights = 1.0 / class_counts
    weights = weights / weights.sum()
    return torch.tensor(weights, dtype=torch.float32)
    

device = "cuda"
from torch.utils.data import DataLoader
train_loader = DataLoader(
        train_dataset,
        batch_size=8,
        num_workers=4,
        shuffle=True,
        pin_memory=True,
    )

val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False, num_workers=0)

test_loader  = DataLoader(test_dataset,  batch_size=8, shuffle=False, num_workers=0)


model = UnifiedBackbone(model_name = model_name)
class_weights = compute_class_weights(train_df).to(device)
print(class_weights)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.05)


tensor([0.9638, 0.0362], device='cuda:0')


In [24]:

best_model = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    epochs=20,
    patience=3,
    str_prefix=str_prefix
)




--- Epoch 1/20 ---


Val: 100%|█████████████████████████████████████████████████████████████████████████████| 84/84 [00:23<00:00,  3.55it/s]


train loss 0.4019
Train Metrics:
  accuracy: 0.92
  balanced_accuracy: 0.66
  f1: 0.96
  roc_auc: 0.80
  auprc: 0.99
  conf_matrix:
[[  41   64]
 [ 170 2629]]
val loss 0.2970

Val Metrics:
  accuracy: 0.96
  balanced_accuracy: 0.69
  f1: 0.98
  roc_auc: 0.91
  auprc: 0.99
  conf_matrix:
[[ 11  16]
 [ 14 627]]
→ Best model updated.

--- Epoch 2/20 ---


Train:   0%|                                                                                   | 0/363 [00:07<?, ?it/s]


KeyboardInterrupt: 

In [25]:
from img_quality_train_val import test
import torch
import pandas as pd
from torch.utils.data import DataLoader
from diagnosis_train_eval import train_model as train_model_diagnosis
from diagnosis_train_eval import validate as validate_diagnosis
from diagnosis_train_eval import evaluate_thresholds
from model import UnifiedBackbone

df_out=None
device = "cuda"
best_model_img_q = UnifiedBackbone(model_name = model_name).cuda()
best_model_img_q.load_state_dict(torch.load(str_prefix+"img_quality_model_392.pth"))



val_loss, val_metrics = validate(best_model_img_q, val_loader, loss_fn, device)
recall_targets = [0.99, 0.95, 0.75, 0.55, 0.35, 0.15, 0.05, 0.01]
thresholds = find_thresholds_for_recall(val_metrics['all_labels'], val_metrics['all_probs'], recall_targets)


best_model_diagnosis = UnifiedBackbone(model_name = model_name).cuda()
if dataset == "mBRSET":
    best_model_diagnosis.load_state_dict(torch.load("mBRSET_img_diagnosis_model_392.pth"))
    # Filter the test dataset for this model with images that got through image quality checker.
    test_df = pd.read_pickle(data_dir + "data\\mbrset_icdr_quality_2826_test_full.pkl")
    test_df = (
        test_df  # optional but recommended for determinism
        .groupby("patient", as_index=False)
        .first()
    )

if dataset == "BRSET":
    best_model_diagnosis.load_state_dict(torch.load("BRSET_img_diagnosis_model_392.pth"))
    # Filter the test dataset for this model with images that got through image quality checker.
    
    test_df = pd.read_pickle(data_dir + "data\\brset_test_2826.pkl")
    test_df = test_df.rename(columns={"patient_id": "patient"})
    
    test_df = (
        test_df  # optional but recommended for determinism
        .groupby("patient", as_index=False)
        .first()
    )
test_dataset  = FundusDataset(test_df,  img_root=img_root, high_quality_tf=val_tf, low_quality_tf = val_tf,label_col='final_quality')
test_loader  = DataLoader(test_dataset,  batch_size=8, shuffle=False, num_workers=0)

print('Test: Before',len(test_df))

for target, thr in thresholds["class_0"].items():
    print(f"Recall {target*100:.0f}% → threshold = {thr:.4f}") 
    loss1, metrics1, good_quality_file_names = test(best_model_img_q, test_loader, loss_fn, device,T=thr)
    print("test loss", format(loss1, ".4f"))
    print("\nTest Metrics:")
    for k, v in metrics1.items():
    # Case 1: confusion matrix or any array-like
        if isinstance(v, np.ndarray):
            print(f"  {k}:")
            print(v)
    # Case 2: normal number
        else:
            print(f"  {k}: {format(v, '.2f')}")
    output_metrics={}
    output_metrics['img_quality_th'] = thr
    output_metrics['recall_low_quality_target'] = target
    output_metrics['recall_low_quality_actual'] = metrics1['conf_matrix'][0][0]/(metrics1['conf_matrix'][0][0]+metrics1['conf_matrix'][0][1])


    test_df_modified = test_df[test_df['file'].isin(good_quality_file_names)]
    print('Test:After',len(test_df_modified))
    test_ds_modified = FundusDataset(test_df_modified,img_root, high_quality_tf=val_tf, low_quality_tf=val_tf, label_col='final_icdr')
    test_loader_modified  = DataLoader(test_ds_modified,  batch_size=16, shuffle=False, num_workers=2)
    
    loss1, metrics1,all_files, corr_files, incorrect_files = validate_diagnosis(best_model_diagnosis, test_loader_modified, loss_fn, device)
    print("test loss", format(loss1, ".4f"))
    print("\nTest Metrics:")
    for k, v in metrics1.items():
    # Case 1: confusion matrix or any array-like
        if isinstance(v, np.ndarray):
            print(f"  {k}:")
            print(v)
    # Case 2: normal number
        elif not isinstance(v,list):
            print(f"  {k}: {format(v, '.2f')}")

    for key in ['ba','f1']:
        output_metrics[key] = metrics1[key]

    output_metrics['n_samp'] = len(test_df)
    output_metrics['n_conf'] = len(test_df_modified)
    output_metrics['coverage'] =     output_metrics['n_conf']/output_metrics['n_samp']
    if df_out is None:
        df_out = pd.DataFrame([output_metrics])
    else:
        df_out = pd.concat([df_out, pd.DataFrame([output_metrics])], ignore_index=True)
    print(df_out.head())

Val: 100%|█████████████████████████████████████████████████████████████████████████████| 84/84 [00:23<00:00,  3.58it/s]


Test: Before 157
Recall 99% → threshold = 0.9750


Test: 100%|████████████████████████████████████████████████████████████████████████████| 20/20 [00:05<00:00,  3.41it/s]


test loss 0.3283

Test Metrics:
  accuracy: 0.19
  balanced_accuracy: 0.57
  f1: 0.25
  roc_auc: 0.95
  auprc: 1.00
  conf_matrix:
[[  9   0]
 [127  21]]
Test:After 21


Val: 100%|███████████████████████████████████████████████████████████████████████████████| 2/2 [00:06<00:00,  3.24s/it]


test loss 0.0734

Test Metrics:
  accuracy: 0.95
  ba: 0.88
  f1: 0.86
  roc_auc: 0.99
  auprc: 0.95
  conf_matrix:
[[17  0]
 [ 1  3]]
   img_quality_th  recall_low_quality_target  recall_low_quality_actual   
0        0.974987                       0.99                        1.0  \

      ba        f1  n_samp  n_conf  coverage  
0  0.875  0.857143     157      21  0.133758  
Recall 95% → threshold = 0.9100


Test:  50%|██████████████████████████████████████                                      | 10/20 [00:02<00:02,  3.71it/s]


KeyboardInterrupt: 

In [ ]:
data_result = r"C:\Users\preet\Documents\mBRSET\mBRSET_image_quality\results\mbrset_results"
df_out.to_csv(data_result + "img_quality_check_cascaded_"+ dataset + "_retfound_greeen_21426_" + str(run_id) + ".csv")